# Bead-pull calibration

Map the **stepper-motor step count** to the **bead position** for every pass of
the thread through the booster (each pass = one *sub-thread*).

For each sub-thread you record, at both ends, two things:

* the **motor step count** &mdash; **start** (the bead-position zero of that
  sub-thread) and **end** (the far end of the pass), and
* the **(x, y, z) position in the booster frame** &mdash; `start_xyz_m` and
  `end_xyz_m`.

From those the code derives everything else per sub-thread: the travel
*direction* and step *length* from the step anchors, the physical *length* from
the distance between the two 3-D endpoints, and the **step&harr;metre
conversion** from the two together (step span &divide; physical length).  There
is **no single global `steps_per_meter`** any more &mdash; each sub-thread carries
its own, so nothing extra has to be measured or entered.

> Because the 3-D endpoints now set the scale, their coordinates must be
> accurate: the distance between `start_xyz_m` and `end_xyz_m` *is* the physical
> length the step span is divided by.

This rig has **no limit switch**, so there is no automatic hardware home.
Instead you set a **home position** yourself (Section 4): jog the bead to a
repeatable physical reference and store it.  The current motor position *and* the
home position are saved to a shared file at a fixed absolute path
(`~/.madmax_bead_pull/motor_state.json`), so they survive power cycles and every
program on this computer uses the same reference.  Stepper settings come from the
shared `config/measurement_config.json`.

This notebook produces the **calibration file**: per sub-thread its `index`,
`name`, the non-measurement margins, the **(x, y, z) start/end points**, and the
measured `start_steps` / `end_steps`.

**Workflow:** run the cells top-to-bottom.  Set the **home position** (Section 4)
once at the start, then Section 5 is the loop you repeat for each sub-thread: jog
the bead to the start, capture it; jog to the end, capture it.

## 1. Imports and configuration

In [ ]:
import json
import sys
from pathlib import Path

# Make ``src`` importable (this notebook lives at the repo root).
sys.path.insert(0, str(Path.cwd()))

from src.bead_pull_controller import (
    Calibration,
    SubThreadCalibration,
    Stepper,
    SimulatedMotor,
    motor_state_path,
)

# ---- edit these -----------------------------------------------------------
SIMULATE = True          # True: no hardware (dry test of the flow). False: real motor.

# The passes of the thread through the booster you will calibrate, in order.
# For each pass set:
#   index          : integer id
#   name           : label (shown in logs)
#   margin_start_m : skip this many metres after the start (no measurement there)
#   margin_end_m   : skip this many metres before the end (no measurement there)
#   start_xyz_m    : (x, y, z) of the START point in the booster frame, metres
#   end_xyz_m      : (x, y, z) of the END point in the booster frame, metres
# start_xyz_m / end_xyz_m are REQUIRED: the distance between them is the physical
# length that fixes this sub-thread's step<->metre scale, so enter them
# accurately.  The step anchors (start_steps / end_steps) are captured below by
# jogging.
SUB_THREADS = [
    {"index": 0, "name": "z0", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [7.5, 19., 17.7], "end_xyz_m": [-7.5, -19., 17.7]},
    {"index": 1, "name": "z1", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [7., -19., 18.7], "end_xyz_m": [-7., 19., 18.7]},
    {"index": 2, "name": "z2", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [7.5, 19., 19.7], "end_xyz_m": [-7.5, -19., 19.7]},
    {"index": 3, "name": "z3", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [7., -19., 20.7], "end_xyz_m": [-7., 19., 20.7]},
]

# ---- from the shared measurement config -----------------------------------
# Stepper settings and the calibration-file path live here.
MEASUREMENT_CONFIG = Path("config/measurement_config.json")
_cfg = json.loads(MEASUREMENT_CONFIG.read_text(encoding="utf-8"))
STEPPER = _cfg.get("stepper", {})
CALIBRATION_PATH = Path(_cfg.get("calibration_file", "config/bead_pull_calibration.json"))

print(f"Stepper: port={STEPPER.get('port')}, baud={STEPPER.get('baud')}, motor={STEPPER.get('motor')}")
print(f"sub-threads to calibrate: {[d['index'] for d in SUB_THREADS]}")
print(f"Calibration will be saved to: {CALIBRATION_PATH}")

## 2. Connect to the motor

In [ ]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()

for port in ports:
    print(f"Port: {port.device}")
    print(f"Description: {port.description}")
    print(f"Hardware ID: {port.hwid}\n")

In [ ]:
if SIMULATE:
    motor = SimulatedMotor(verbose=True)
    print("SIMULATE = True: using an in-memory motor (no serial port opened).")
else:
    motor = Stepper.open(**STEPPER)   # the config keys match Stepper.open's arguments
    print("Connected to", STEPPER.get("port"))

## 2b. Emergency stop &mdash; halt the motor at any time

Run this cell **right after connecting** so a stop is always within reach while
you jog. It gives you two independent ways to halt the motor **immediately, even
mid-move**:

* **The red `EMERGENCY STOP` button.** It stays clickable while the motor is
  moving *provided the move was started with `jog_bg(...)`* (which runs the move
  in the background so the button's click is serviced). One click sends
  `hardstop` + `hiz`.
* **`Kernel ▸ Interrupt`** (the ■ button in the toolbar). This works during *any*
  running cell &mdash; including an ordinary blocking `jog(...)` &mdash; and the
  controller converts the interrupt into the same `hardstop` + `hiz`.

Both stop the motor instantly and switch the coils off. Because a hard stop
loses steps, **the position is unreliable afterwards**: bring the bead to a known
reference and call `motor.set_position(<steps>)` before moving on.

> Use `jog(...)` (Section 3) for normal, blocking jogs &mdash; those are still
> stoppable via `Kernel ▸ Interrupt`. Use `jog_bg(...)` when you specifically
> want the on-screen button to work during the move (e.g. a long traverse).


In [ ]:
# --- Emergency stop -------------------------------------------------------
# Two INDEPENDENT ways to halt the motor at ANY time, even while it is moving:
#
#   1. The red EMERGENCY STOP button below. It stays clickable while the motor
#      is moving IF you launched the move with jog_bg(...) (which runs the move
#      in the background). Clicking it sends hardstop + hiz immediately.
#
#   2. Kernel > Interrupt (the square stop button in the toolbar). This works
#      during ANY running cell -- including a plain synchronous jog(...) -- and
#      the controller turns that interrupt into the same hardstop + hiz.
#
# Either way the motor stops at once and its coils are switched off. AFTER an
# emergency stop the absolute position is unreliable (a hard stop loses steps):
# put the bead at a known reference and call motor.set_position(<steps>) before
# trusting positions or moving again.
import threading


def estop():
    """Halt the motor immediately: hardstop + coils off. Callable any time."""
    motor.emergency_stop()


_jog_thread = None


def jog_bg(steps):
    """Jog by ``steps`` in the BACKGROUND so the STOP button stays responsive.

    Returns immediately; the move runs in a worker thread. Use moving() to check
    whether it is still going and wait_move() to block until it finishes. To stop
    it early, click EMERGENCY STOP (or call estop()).
    """
    global _jog_thread
    if moving():
        print("A background jog is already running; wait for it, or hit STOP.")
        return
    steps = int(steps)

    def _run():
        try:
            motor.move_by(steps)
            print(f"jog_bg {steps:+d} done -> {motor.get_position()} steps")
        except Exception as exc:  # EmergencyStop, serial errors, ...
            print(f"jog_bg {steps:+d} halted: {type(exc).__name__}: {exc}")

    _jog_thread = threading.Thread(target=_run, daemon=True)
    _jog_thread.start()
    print(f"jog_bg {steps:+d} started (background). Press EMERGENCY STOP any time.")


def moving():
    """True while a background jog is still running."""
    return _jog_thread is not None and _jog_thread.is_alive()


def wait_move():
    """Block (interruptibly) until the background jog finishes."""
    if _jog_thread is not None:
        _jog_thread.join()


# On-screen red button (optional; needs ipywidgets). If it is unavailable, the
# Kernel > Interrupt route and estop() still work.
try:
    import ipywidgets as widgets
    from IPython.display import display

    _stop_btn = widgets.Button(
        description="EMERGENCY STOP",
        button_style="danger",
        icon="hand-paper-o",
        tooltip="Immediately stop the motor and switch off its coils",
        layout=widgets.Layout(width="260px", height="64px"),
    )
    _stop_btn.style.font_weight = "bold"
    _stop_btn.on_click(lambda _btn: estop())
    display(_stop_btn)
    print("Red EMERGENCY STOP button ready (works during jog_bg moves).")
except Exception as _exc:  # ipywidgets missing / no frontend
    print(f"(No on-screen button: {_exc}.)")
    print("Use estop() or Kernel > Interrupt to stop the motor.")

print("Emergency-stop ready: estop(), jog_bg(steps), moving(), wait_move().")


## 3. Jog helpers

`jog(steps)` moves the bead by a relative number of steps (positive vs. negative
= the two directions; start with small values and watch which way the bead goes)
and prints the new absolute position.  `pos()` just prints the current position.
Use these to bring the bead to a reference: the **home position** (Section 4) and
each sub-thread's start/end (Section 5).

In [ ]:
def jog(steps):
    """Move the bead by a relative number of steps and report the new position."""
    motor.move_by(int(steps))
    p = motor.get_position()
    print(f"jogged {int(steps):+d} -> {p} steps")
    return p

def pos():
    p = motor.get_position()
    print("position:", p, "steps")
    return p

print("Helpers ready: jog(steps), pos().")

### Jog cell — run/edit this as many times as you need

Use this to drive the bead wherever you need it next (the home reference in
Section 4, or a sub-thread endpoint in Section 5).

In [ ]:
# Examples (edit the number, re-run):
jog(500)
# jog(-200)
# pos()

## 4. Set the home position

There is **no limit switch**, so *you* define home.  Pick a repeatable physical
reference for the bead, **jog to it** (Section 3 / the jog cell above), then run
the **set-home** cell below to store it.

The home position **and** the current motor position are written to a shared file
at a fixed absolute path (shown below), so they survive power cycles and every
program on this computer — this notebook, the measurement script, anything else —
uses the same reference.  `motor.home()` later moves the bead straight back to
this saved position (a bounded move, never the old runaway to a switch).

> **If the bead was moved by hand while the controller was off**, the saved
> position is stale.  Tell the controller where the bead actually is first, e.g.
> `motor.set_position(0)` (declares "the bead is at step 0 right now"), then set
> home.

In [ ]:
# Where the shared state lives, plus the current position and stored home.
print("Current position:", motor.get_position(), "steps")
print("Stored home position:", motor.home_position)
if SIMULATE:
    print("(SIMULATE: state is in-memory only; the real motor persists to",
          motor_state_path(), ")")
else:
    print("Shared motor-state file:", motor_state_path())

In [ ]:
# Set home. With the bead physically at your chosen HOME reference (jog to it
# above), run this to store the CURRENT position as home:
h = motor.set_home()
print("Home position set to", h, "steps.")
if not SIMULATE:
    print("Saved to the shared state file:", motor_state_path())

# Alternatives:
#   motor.set_home(12345)   # store a specific step count as home
#   motor.set_position(0)   # re-declare the current position (do this first if
#                           # the bead was moved by hand while powered off)

## 5. Capture each sub-thread's start / end

Using `jog(...)` / `pos()` from section 3, for **each** sub-thread:

1. `jog(...)` until the bead sits at the sub-thread's **start** (zero), then run
   `capture(index, "start")`.
2. `jog(...)` until the bead sits at the sub-thread's **end**, then run
   `capture(index, "end")`.

You can revisit any sub-thread and capture again to overwrite a bad value.

In [ ]:
captured = {}   # index -> {"start_steps": ..., "end_steps": ...}

def capture(index, which):
    """Record the current position as the start/end of a sub-thread."""
    which = which.lower()
    assert which in ("start", "end"), "which must be 'start' or 'end'"
    p = motor.get_position()
    captured.setdefault(index, {})[f"{which}_steps"] = p
    print(f"sub-thread {index}: {which} = {p} steps")
    return p

print("Helper ready: capture(index, 'start'|'end').  (jog/pos come from section 3.)")

### Capture cells — run when the bead is at the start / end of a sub-thread

In [ ]:
jog_bg(200)

In [ ]:
# When the bead is at the START (zero) of, say, sub-thread 0:
capture(0, "start")


In [ ]:
# When the bead is at the END of sub-thread 0:
capture(0, "end")


Repeat the jog + capture cells for every entry in `SUB_THREADS` (indices
`1`, `2`, ... here).  You can copy the two capture lines and change the index, or
just re-run them after editing the index.  Check progress any time:

In [ ]:
for d in SUB_THREADS:
    i = d["index"]
    got = captured.get(i, {})
    print(f"sub-thread {i} ({d['name']}): "
          f"start={got.get('start_steps', '---')}, end={got.get('end_steps', '---')}")


> **Tip for a dry test (`SIMULATE = True`):** there is no real bead, so just move
> the simulated motor before capturing, e.g. `motor.move_by(0); capture(0,'start')`
> then `motor.move_by(20000); capture(0,'end')`, to exercise the save below.

## 6. Build and save the calibration

In [ ]:
sub_threads = []
for d in SUB_THREADS:
    i = d["index"]
    if i not in captured or "start_steps" not in captured[i] or "end_steps" not in captured[i]:
        raise ValueError(f"sub-thread {i} ({d.get('name')}) is missing a start or end capture.")
    if d.get("start_xyz_m") is None or d.get("end_xyz_m") is None:
        raise ValueError(f"sub-thread {i} ({d.get('name')}) is missing start_xyz_m / end_xyz_m.")
    sub_threads.append(
        SubThreadCalibration(
            index=i,
            name=d.get("name"),
            margin_start_m=d.get("margin_start_m", 0.0),
            margin_end_m=d.get("margin_end_m", 0.0),
            start_xyz_m=d["start_xyz_m"],
            end_xyz_m=d["end_xyz_m"],
            start_steps=captured[i]["start_steps"],
            end_steps=captured[i]["end_steps"],
        )
    )

# Each sub-thread's step<->metre scale is derived from its own anchors (step span
# / distance between the 3-D endpoints); the summary's steps/m column shows it.
calibration = Calibration(sub_threads=sub_threads)
print(calibration.summary())
path = calibration.save(CALIBRATION_PATH)
print("\nSaved calibration to:", path)

## 7. Verify (reload from disk)

In [ ]:
# Reload the saved calibration (anchors + name + 3-D endpoints + margins),
# exactly as run_bead_pull_measurement.py does at load time.  The step<->metre
# scale is derived per sub-thread from the anchors, so nothing else is passed.
reloaded = Calibration.from_calibration_file(CALIBRATION_PATH)
print(reloaded.summary())

## 8. Release the motor

Switches the coils off and closes the serial port (no-op for the simulated
motor).

In [ ]:
motor.shutdown()
print("Motor released.")
